# 第 3 章 向量化运算与广播机制 (Vectorization & Broadcasting)

> 🎯 **本章目标**
>
> - 理解「逐元素运算」：为什么 `arr + arr` 是加法，而 `list + list` 是拼接
> - 掌握 ufunc（通用函数）的概念与常用成员
> - 吃透**广播（Broadcasting）**三规则，看懂任意形状如何对齐
> - 学会用比较运算生成布尔数组，并与第 2 章的布尔索引衔接
> - 用实测数据感受「向量化」为何比 Python 循环快几十到上千倍
> - 区分原地运算与赋值运算的内存差异

> 📖 前置知识：第 1 章 ndarray 基础与 dtype、第 2 章索引切片与布尔筛选。

本章我们终于要告别「Python 循环 + 逐个计算」的写法，真正开始用 NumPy 的思维方式思考问题。

In [1]:
import numpy as np
print(np.__version__)

2.4.4


## 3.1 逐元素运算：数组相加 vs 列表相加

### 为什么先讲这个？

很多初学者第一次接触 NumPy 时最困惑的问题就是：
「`arr + arr` 到底在算什么？」

答案是——**逐元素（element-wise）**：对应位置的元素两两相加。
而 Python 原生的 `list + list` 是**拼接**，把两个列表首尾连起来。

这是数组与列表最本质的区别之一，也是理解后面所有内容（ufunc、广播）的地基。

| 操作 | `list + list` | `arr + arr` (NumPy) |
| --- | --- | --- |
| 含义 | 拼接（顺序连接） | 逐元素相加 |
| `[1,2,3] + [10,20,30]` | `[1,2,3,10,20,30]` | `[11,22,33]` |
| 是否要求等长 | 不要求 | 要求（或可广播） |
| 返回值类型 | `list` | `ndarray` |

> 💡 **记忆口诀**：列表是「盒子串」，`+` 是把盒子串接起来；数组是「数字表格」，`+` 是对应格子里的数相加。

```mermaid
flowchart LR
    A["列表 [1,2,3]"] -->|"+"| B["拼接 → [1,2,3,1,2,3]"]
    C["数组 [1,2,3]"] -->|"+"| D["逐元素 → [2,4,6]"]
```

In [2]:
# 列表相加 = 拼接, 数组相加 = 逐元素
list_a = [1, 2, 3]
list_b = [10, 20, 30]
print("list + list =", list_a + list_b)   # 结果是拼接, 不是数学加法!

arr_a = np.array([1, 2, 3])
arr_b = np.array([10, 20, 30])
arr_sum = arr_a + arr_b
print("arr + arr =", arr_sum)
print("arr_a.shape =", arr_a.shape, "| arr_sum.shape =", arr_sum.shape)

list + list = [1, 2, 3, 10, 20, 30]
arr + arr = [11 22 33]
arr_a.shape = (3,) | arr_sum.shape = (3,)


In [3]:
# 逐元素运算全家桶: 加减乘除、整除、取余、幂
a = np.array([1, 2, 3, 4])
b = np.array([2, 3, 4, 5])

print("a + b  =", a + b)       # 加法
print("a - b  =", a - b)       # 减法
print("a * b  =", a * b)       # 逐元素相乘（不是矩阵乘法!）
print("a / b  =", a / b)       # 除法
print("a // b =", a // b)      # 整除
print("a % b  =", a % b)       # 取余
print("a ** b =", a ** b)      # 幂
print("结果形状:", (a + b).shape)

a + b  = [3 5 7 9]
a - b  = [-1 -1 -1 -1]
a * b  = [ 2  6 12 20]
a / b  = [0.5        0.66666667 0.75       0.8       ]
a // b = [0 0 0 0]
a % b  = [1 2 3 4]
a ** b = [   1    8   81 1024]
结果形状: (4,)


> ⚠️ **陷阱**：`*` 是逐元素相乘，**不是**矩阵乘法！矩阵乘法是 `a @ b` 或 `np.matmul(a, b)`，第 5 章会讲。

## 3.2 数组与标量运算：标量自动「扩散」

### 解决什么问题？

你不需要写循环去「把每个元素都加 10」。
NumPy 会把标量自动**扩散（broadcast）**到数组的每一个元素上：

```python
arr = np.array([1, 2, 3, 4])
arr + 10   # 等价于手动逐元素 [1+10, 2+10, 3+10, 4+10]
```

> 💡 这里的直觉就是广播的雏形：**小形状自动扩展到大形状**。下一节我们会把它形式化成三条规则。

```mermaid
flowchart LR
    S["标量 10"] -->|"自动扩散"| E["每个元素都参与运算"]
    A["[1, 2, 3, 4]"] --> E
    E --> R["[11, 12, 13, 14]"]
```

In [4]:
# 数组 + 标量: 标量自动扩散到每个元素
arr = np.array([1, 2, 3, 4])
print("arr + 10 =", arr + 10)     # 每个元素都加 10
print("arr * 2  =", arr * 2)      # 每个元素都乘 2
print("arr ** 2 =", arr ** 2)     # 每个元素都平方

# 二维数组同样适用
mat = np.arange(6).reshape(2, 3)
print("\nmat =", mat, "| mat.shape =", mat.shape)
print("mat + 1 =", mat + 1)

arr + 10 = [11 12 13 14]
arr * 2  = [2 4 6 8]
arr ** 2 = [ 1  4  9 16]

mat = [[0 1 2]
 [3 4 5]] | mat.shape = (2, 3)
mat + 1 = [[1 2 3]
 [4 5 6]]


## 3.3 ufunc（通用函数）是什么？

### 为什么要有这个名字？

你看到的 `+ - * /` 这些运算符，其实都是**函数**的「语法糖」。
NumPy 把「对每个元素做相同运算」的函数统称为 **ufunc（universal function，通用函数）**。

ufunc 的特别之处：**在 C 语言层面循环**，而不是在 Python 层面循环。
这就解释了为什么它快——后面 3.6 会实测。

### 分类

| 类型 | 含义 | 例子 |
| --- | --- | --- |
| 一元 ufunc | 输入 1 个数组，输出 1 个数组 | `np.sqrt` `np.exp` `np.log` `np.abs` `np.square` `np.floor` `np.ceil` `np.round` |
| 二元 ufunc | 输入 2 个数组，输出 1 个数组 | `np.add` `np.multiply` `np.maximum` `np.minimum` `np.fmod` `np.power` |

> 💡 运算符与 ufunc 是一一对应的：`a + b` ≡ `np.add(a, b)`，`a * b` ≡ `np.multiply(a, b)`。

In [5]:
# 一元 ufunc: 一个数组进, 一个数组出
arr = np.array([1, 4, 9, 16])
print("np.sqrt(arr) = ", np.sqrt(arr))                  # 开平方
print("np.square(arr)=", np.square(arr))                # 平方
print("np.abs(np.array([-1,2,-3])) =", np.abs(np.array([-1, 2, -3])))  # 绝对值
print("np.log(np.array([1, np.e])) =", np.log(np.array([1, np.e])))    # 自然对数
print("np.exp(np.array([0., 1.])) =", np.exp(np.array([0., 1.])))      # e 的幂

np.sqrt(arr) =  [1. 2. 3. 4.]
np.square(arr)= [  1  16  81 256]
np.abs(np.array([-1,2,-3])) = [1 2 3]
np.log(np.array([1, np.e])) = [0. 1.]
np.exp(np.array([0., 1.])) = [1.         2.71828183]


In [6]:
# 二元 ufunc: 两个数组进, 一个数组出
a = np.array([1, 2, 3])
b = np.array([2, 3, 4])

print("np.add(a, b)      =", np.add(a, b))        # 等价于 a + b
print("np.multiply(a, b) =", np.multiply(a, b))   # 等价于 a * b
print("np.maximum(a, b)  =", np.maximum(a, b))    # 逐元素取较大者
print("np.minimum(a, b)  =", np.minimum(a, b))    # 逐元素取较小者
print("np.fmod(a, b)     =", np.fmod(a, b))       # 逐元素取余(符号随被除数)

np.add(a, b)      = [3 5 7]
np.multiply(a, b) = [ 2  6 12]
np.maximum(a, b)  = [2 3 4]
np.minimum(a, b)  = [1 2 3]
np.fmod(a, b)     = [1 2 3]


## 3.4 广播（Broadcasting）规则详解

### 解决什么问题？

不同形状的数组也能做逐元素运算——前提是它们「兼容」。
广播就是 NumPy 自动把较小的形状**拉伸**到较大形状的机制，省去你手动复制数据。

### 三条规则（背下来）

1. **对齐**：从最右边的维度开始，逐个向左对齐（`shape` 元组从后往前看）
2. **补 1**：维度数不够的数组，左边补上长度为 1 的虚拟维度
3. **拉伸**：对齐后，对应维度要么**相等**，要么其中一个是 **1**（1 会被拉伸成对方的长度）；否则报错

### 例子：(3, 1) + (1, 4) → (3, 4)

```mermaid
flowchart TD
    A["数组 A 形状 (3, 1)"] -->|"右对齐"| R["广播判定"]
    B["数组 B 形状 (1, 4)"] -->|"右对齐"| R
    R -->|"规则: 对应维相等或为1"| S["A 沿 axis=1 拉伸 4 次 → (3,4)<br/>B 沿 axis=0 拉伸 3 次 → (3,4)"]
    S --> O["结果形状 (3, 4)"]
    O --> P["每个位置: A[i,0] + B[0,j]"]
```

> 💡 **直觉**：把 (3,1) 想象成「3 行 × 1 列」的竖条，把 (1,4) 想象成「1 行 × 4 列」的横条。竖条向右铺 4 份、横条向下铺 3 份，就叠出一个 3×4 的棋盘。

In [7]:
# 广播成功案例 1: (3,1) + (1,4) -> (3,4)
A = np.array([[1], [2], [3]])          # 形状 (3, 1)
B = np.array([[10, 20, 30, 40]])       # 形状 (1, 4)
C = A + B
print("A.shape =", A.shape, "| B.shape =", B.shape)
print("结果 C.shape =", C.shape)
print("C =\n", C)

A.shape = (3, 1) | B.shape = (1, 4)
结果 C.shape = (3, 4)
C =
 [[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


In [8]:
# 广播成功案例 2: (3,) 与 (2,3) 对齐 -> (2,3)
col = np.array([1, 2, 3])              # 形状 (3,)
mat = np.arange(6).reshape(2, 3)       # 形状 (2, 3)
print("col.shape =", col.shape, "| mat.shape =", mat.shape)
print("mat + col =\n", mat + col)      # col 自动被当作行向量, 广播到每一行

col.shape = (3,) | mat.shape = (2, 3)
mat + col =
 [[1 3 5]
 [4 6 8]]


In [9]:
# 广播失败案例: (3,2) + (3,) 为什么报错?
a = np.arange(6).reshape(3, 2)         # 形状 (3, 2)
b = np.array([1, 2, 3])                # 形状 (3,)
try:
    print(a + b)
except ValueError as e:
    print("报错:", e)

# 原因分析: 从右往左对齐
#   a: (3, 2)
#   b:    (3,)  -> 补1后 (1, 3)
#   对齐: 最右边 2 vs 3 → 不相等且都不为 1 → 广播失败!

报错: operands could not be broadcast together with shapes (3,2) (3,) 


In [10]:
# np.broadcast_shapes: 官方"裁判", 预先判断哪些形状能广播
print(np.broadcast_shapes((3, 1), (1, 4)))   # -> (3, 4)
print(np.broadcast_shapes((3,), (2, 3)))     # -> (2, 3)
print(np.broadcast_shapes((5,), (1,)))       # -> (5,)
print(np.broadcast_shapes((3, 4), (1, 4)))   # -> (3, 4)

try:
    np.broadcast_shapes((3, 2), (3,))        # 会抛异常
except ValueError as e:
    print("(3,2) 与 (3,) 无法广播:", e)

(3, 4)
(2, 3)
(5,)
(3, 4)
(3,2) 与 (3,) 无法广播: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (3, 2) and arg 1 with shape (3,).


## 3.5 比较运算：返回布尔数组

### 解决什么问题？

`arr > 3` 不是返回一个 True/False，而是返回**一个和 arr 形状相同的布尔数组**，
每个位置记录「该元素是否大于 3」。

这正是第 2 章「布尔索引」的原料：布尔数组可以直接当作索引来筛选元素。

```python
arr = np.array([1, 5, 3, 9, 2])
arr > 3        # array([False,  True, False,  True, False])
arr[arr > 3]   # array([5, 9])  布尔索引
```

### 两个常用聚合

- `np.all(条件)`：全部满足才为 True
- `np.any(条件)`：至少一个满足就为 True

> ⚠️ **陷阱**：比较**两个数组是否相等**，**不能**用 `a == b` 当布尔值用——
> `a == b` 得到的是布尔数组，`if (a == b)` 会报错「布尔值不明确」！
> 应该用 `np.array_equal(a, b)`。

In [11]:
# 比较运算逐元素比较, 返回布尔数组
arr = np.array([1, 5, 3, 9, 2])
print("arr > 3  =", arr > 3)
print("arr == 3 =", arr == 3)
print("arr != 3 =", arr != 3)
print("arr > 3 的形状:", (arr > 3).shape)

# 与第2章布尔索引衔接: 布尔数组直接用于筛选
print("arr[arr > 3] =", arr[arr > 3])

arr > 3  = [False  True False  True False]
arr == 3 = [False False  True False False]
arr != 3 = [ True  True False  True  True]
arr > 3 的形状: (5,)
arr[arr > 3] = [5 9]


In [12]:
# np.all / np.any: 判断"全部"还是"存在"
arr = np.array([1, 5, 3, 9, 2])
print("np.all(arr > 0) =", np.all(arr > 0))   # 全部 > 0 ? -> True
print("np.all(arr > 3) =", np.all(arr > 3))   # 全部 > 3 ? -> False
print("np.any(arr > 8) =", np.any(arr > 8))   # 存在 > 8 ? -> True

np.all(arr > 0) = True
np.all(arr > 3) = False
np.any(arr > 8) = True


In [13]:
# 陷阱: 判断两个数组是否相等, 不能用 == 或 and
a = np.array([1, 2, 3])
b = np.array([1, 2, 3])
print("a == b 得到的是布尔数组:", a == b)   # 不是 True/False!
print("np.array_equal(a, b) =", np.array_equal(a, b))      # 正确方式
print("np.array_equal(a, [1,2,4]) =", np.array_equal(a, np.array([1, 2, 4])))

# 若只是想知道"是否逐元素全相等", 也可以用 (a == b).all()
print("(a == b).all() =", (a == b).all())

a == b 得到的是布尔数组: [ True  True  True]
np.array_equal(a, b) = True
np.array_equal(a, [1,2,4]) = False
(a == b).all() = True


## 3.6 向量化性能实测：为什么快这么多？

### 核心原理

| | Python 双重循环 | NumPy 向量化 |
| --- | --- | --- |
| 在哪算 | Python 解释器逐元素调度 | C 语言层连续内存批量处理 |
| 每次操作 | 解释器解释一遍（开销大） | 整块内存一次遍历 |
| 数据存放 | 不连续、含类型检查 | 连续内存、类型已知 |

```mermaid
flowchart TD
    subgraph PY["Python 双重循环"]
        P1["for i in range(n)"] --> P2["for j in range(n)"]
        P2 --> P3["每次访问 data[i,j] 都要: 边界检查 + 类型检查 + 解释器调度"]
    end
    subgraph NP["NumPy 向量化"]
        N1["data ** 2 + np.sum"] --> N2["C 层对连续内存整块批量计算"]
    end
    P3 -->|"慢: 每个元素都付解释器开销"| R["结果一样"]
    N2 -->|"快: 一次调用处理整个数组"| R
```

> 💡 **结论先行**：同样的计算，向量化通常比 Python 循环快**几十到上千倍**。数组越大，差距越明显。

In [14]:
# 实测: 求 1000x1000 数组所有元素的平方和
import time
n = 1000
data = np.random.rand(n, n)

# 方式一: Python 双重循环
start = time.perf_counter()
total_loop = 0.0
for i in range(n):
    for j in range(n):
        total_loop += data[i, j] * data[i, j]
t_loop = time.perf_counter() - start

# 方式二: NumPy 向量化 (逐元素平方 + 聚合求和)
start = time.perf_counter()
total_vec = np.sum(data ** 2)
t_vec = time.perf_counter() - start

print(f"双重循环耗时: {t_loop:.3f} 秒, 结果 = {total_loop:.6f}")
print(f"向量化耗时:   {t_vec:.4f} 秒, 结果 = {total_vec:.6f}")
print(f"加速比: 约 {t_loop / t_vec:.0f} 倍")
print("两种方式结果一致:", np.isclose(total_loop, total_vec))

双重循环耗时: 0.234 秒, 结果 = 333765.461223
向量化耗时:   0.0031 秒, 结果 = 333765.461223
加速比: 约 76 倍
两种方式结果一致: True


## 3.7 原地运算：`+=` / `*=` 与赋值运算的区别

### 解决什么问题？

大数组上，`arr = arr + 1` 会**先创建一块新的内存**，再让变量指向它；
而 `arr += 1` 是**原地修改**，直接改动原数组的内存，不额外分配。

| | `arr += 1`（原地） | `arr = arr + 1`（赋值） |
| --- | --- | --- |
| 是否分配新内存 | 否 | 是 |
| `id(arr)` 是否变化 | 不变 | 变化 |
| 内存占用 | 省 | 需要双倍（旧+新） |
| 是否影响共享该内存的其他引用 | 会 | 不会 |

> ⚠️ **陷阱**：如果多个变量共享同一块内存（比如切片、reshape 视图），
> 对其中一个做 `+=` 会同时改变其他变量看到的数值——因为它们是同一份数据。

In [15]:
# 原地运算 +=: id 不变 -> 没有分配新数组
arr = np.array([1, 2, 3])
print("运算前 id:", id(arr))
arr += 10
print("arr += 10 后:", arr)
print("运算后 id:", id(arr), "| 相同 -> 原地修改, 未分配新数组")

运算前 id: 2445362262064
arr += 10 后: [11 12 13]
运算后 id: 2445362262064 | 相同 -> 原地修改, 未分配新数组


In [16]:
# 赋值运算 = 新数组: id 变化 -> 分配了新的数组
arr2 = np.array([1, 2, 3])
print("运算前 id:", id(arr2))
arr2 = arr2 + 10
print("arr2 = arr2 + 10 后:", arr2)
print("运算后 id:", id(arr2), "| 不同 -> 新建了数组")

运算前 id: 2445362262544
arr2 = arr2 + 10 后: [11 12 13]
运算后 id: 2445362262448 | 不同 -> 新建了数组


In [17]:
# 大数组场景: 原地运算省内存
big = np.ones((2000, 2000))          # float64: 2000*2000*8 字节 ≈ 32 MB
print("big 占用约 %.1f MB" % (big.nbytes / 1024 ** 2))
big *= 2                             # 原地乘 2, 不需要额外 32 MB
print("big[0, 0] =", big[0, 0])
# 对比: big = big * 2 会先申请一块新内存, 旧的等垃圾回收后才释放

big 占用约 30.5 MB
big[0, 0] = 2.0


## 3.8 常用数学 ufunc 速查

### clip：截断到区间

`np.clip(x, 下界, 上界)` 把小于下界的变成下界、大于上界的变成上界，中间不变。

> 💡 对比：不借助 clip 的写法是 `np.minimum(上界, np.maximum(下界, x))`——又绕又难读。clip 就是为这个场景准备的。

### round / floor / ceil / trunc / sign

| 函数 | 作用 | `-1.7` | `1.7` | `-2.3` | `2.3` |
| --- | --- | --- | --- | --- | --- |
| `np.round` | 四舍五入 | -2 | 2 | -2 | 2 |
| `np.floor` | 向下取整 | -2 | 1 | -3 | 2 |
| `np.ceil` | 向上取整 | -1 | 2 | -2 | 3 |
| `np.trunc` | 向零截断 | -1 | 1 | -2 | 2 |
| `np.sign` | 取符号 | -1 | 1 | -1 | 1 |

> ⚠️ **注意**：`np.round(3.5)` 是 4.0，但 `np.round(2.5)` 是 **2.0**——NumPy 采用**银行家舍入**（四舍六入五成双）。

### maximum/minimum 与 max/min 的世纪区别

| | `np.maximum(a, b)` | `np.max(a)` |
| --- | --- | --- |
| 类型 | 二元 ufunc，**逐元素**比较两个数组 | 聚合函数，**整个数组**取一个最大值 |
| 输入 | 两个数组（或数组与标量） | 一个数组 |
| 输出 | 与输入同形状 | 标量（或指定 axis 后的降维结果） |

In [18]:
# clip: 截断到 [5, 15] 区间
arr = np.array([1, 5, 9, 15, 20])
print("np.clip(arr, 5, 15) =", np.clip(arr, 5, 15))
# 等价写法: 用 min/max 组合(繁琐)
print("np.minimum(15, np.maximum(5, arr)) =", np.minimum(15, np.maximum(5, arr)))

np.clip(arr, 5, 15) = [ 5  5  9 15 15]
np.minimum(15, np.maximum(5, arr)) = [ 5  5  9 15 15]


In [19]:
# round / floor / ceil / trunc / sign
x = np.array([1.7, 2.3, -1.7, -2.3, 3.5])
print("x           =", x)
print("np.round(x) =", np.round(x))   # 四舍五入(注意 3.5 的银行家舍入)
print("np.floor(x) =", np.floor(x))   # 向下取整
print("np.ceil(x)  =", np.ceil(x))    # 向上取整
print("np.trunc(x) =", np.trunc(x))   # 向零截断
print("np.sign(x)  =", np.sign(x))    # 符号: -1/0/1

x           = [ 1.7  2.3 -1.7 -2.3  3.5]
np.round(x) = [ 2.  2. -2. -2.  4.]
np.floor(x) = [ 1.  2. -2. -3.  3.]
np.ceil(x)  = [ 2.  3. -1. -2.  4.]
np.trunc(x) = [ 1.  2. -1. -2.  3.]
np.sign(x)  = [ 1.  1. -1. -1.  1.]


In [20]:
# maximum/minimum (逐元素) vs max/min (聚合) —— 重点区分!
a = np.array([[1, 8, 3],
              [7, 2, 6]])
b = np.array([[5, 4, 9],
              [0, 3, 8]])

print("np.maximum(a, b) 逐元素取较大:\n", np.maximum(a, b))   # 逐个位置比较
print("np.max(a) 全局最大:", np.max(a))                        # 整个数组一个最大值
print("np.max(a, axis=0) 每列最大:", np.max(a, axis=0))        # 按列聚合(axis 第4章细讲)
print("np.max(a, axis=1) 每行最大:", np.max(a, axis=1))        # 按行聚合

np.maximum(a, b) 逐元素取较大:
 [[5 8 9]
 [7 3 8]]
np.max(a) 全局最大: 8
np.max(a, axis=0) 每列最大: [7 8 6]
np.max(a, axis=1) 每行最大: [8 7]


## 3.9 实战小案例

### 案例 A：用广播计算欧氏距离矩阵

有 3 个点 P1、P2、P3，想算两两之间的距离，得到 3×3 的距离矩阵。

关键思路：`diff[i, j]` 要等于 `P[i] - P[j]`。用 `pts[:, np.newaxis, :] - pts[np.newaxis, :, :]`：

- `pts[:, None, :]` 形状 `(3, 1, 2)`：每个点是一个「竖块」
- `pts[None, :, :]` 形状 `(1, 3, 2)`：每个点是一个「横块」
- 广播 → `(3, 3, 2)`，再对最后一维求平方和、开方，就是距离矩阵

### 案例 B：数据标准化

`(x - mean) / std`：用每列的均值和标准差对数据做标准化，让每列均值≈0、方差≈1。
这里 `(4, 2)` 的数组减去 `(2,)` 的均值、除以 `(2,)` 的标准差，正是广播的经典应用。

In [21]:
# 案例A: 欧氏距离矩阵
pts = np.array([[0.0, 0.0],     # P1
                [3.0, 4.0],     # P2
                [1.0, 1.0]])    # P3

diff = pts[:, np.newaxis, :] - pts[np.newaxis, :, :]   # (3,1,2)-(1,3,2)->(3,3,2)
dist = np.sqrt(np.sum(diff ** 2, axis=2))              # 对最后一维求和再开方
print("diff.shape =", diff.shape)
print("距离矩阵 (3,3):\n", np.round(dist, 2))
print("P1 到 P2 距离手算验证:", np.sqrt(3 ** 2 + 4 ** 2))   # 应该等于 5.0

diff.shape = (3, 3, 2)
距离矩阵 (3,3):
 [[0.   5.   1.41]
 [5.   0.   3.61]
 [1.41 3.61 0.  ]]
P1 到 P2 距离手算验证: 5.0


In [22]:
# 案例B: 数据标准化 (x - mean) / std
data = np.array([[170, 65],
                 [180, 80],
                 [160, 50],
                 [175, 70]], dtype=float)   # 4 个人: [身高, 体重]
mean = data.mean(axis=0)      # 每列的均值, 形状 (2,)
std  = data.std(axis=0)       # 每列的标准差, 形状 (2,)
norm = (data - mean) / std    # 广播: (4,2) - (2,) -> (4,2), 再除以 (2,)
print("均值:", mean)
print("标准差:", std)
print("标准化后 (均值≈0, 方差≈1):\n", np.round(norm, 4))
print("验证: 标准化后每列均值 ≈", np.round(norm.mean(axis=0), 10))

均值: [171.25  66.25]
标准差: [ 7.39509973 10.82531755]
标准化后 (均值≈0, 方差≈1):
 [[-0.169  -0.1155]
 [ 1.1832  1.2702]
 [-1.5213 -1.5011]
 [ 0.5071  0.3464]]
验证: 标准化后每列均值 ≈ [ 0. -0.]


## 3.10 本章小结

### 一句话记忆表

| 主题 | 一句话 |
| --- | --- |
| 逐元素运算 | 数组运算是对应位置元素运算；`list + list` 才是拼接 |
| ufunc | 在 C 层循环的逐元素函数，运算符只是它的语法糖 |
| 广播 | 右对齐 → 补 1 → 拉伸；对应维度相等或为 1 才兼容 |
| 比较运算 | 返回布尔数组，配合布尔索引 / `all` / `any` 使用 |
| 数组相等 | 用 `np.array_equal`，不要用 `==` / `and` |
| 性能 | 向量化比 Python 循环快几十到上千倍 |
| 原地运算 | `+=` / `*=` 不分配新内存，`arr = arr + 1` 分配新数组 |

```mermaid
mindmap
  root((第3章 向量化))
    逐元素运算
      "数组 vs 列表"
      "四则 整除 取余 幂"
    ufunc
      "一元 sqrt exp log abs"
      "二元 add maximum minimum"
    广播
      "右对齐 补1 拉伸"
      "失败案例 (3,2)+(3,)"
    布尔数组
      "比较运算"
      "all 与 any"
    性能
      "C层批量 快千倍"
      "原地运算省内存"
```

### 📝 动手练习

1. 用广播把「5 个城市的 2016–2020 年人口」数组（形状 `(5, 5)`）的每一行，都减去该城市 2016 年的人口数，得到「相对 2016 年的增长量」。（提示：取第 0 列，用 `[:, np.newaxis]` 变成 `(5, 1)` 再相减。）
2. 给定 `arr = np.array([[1, 2], [3, 4]])`，用 `np.repeat` 把它变成 `[[1, 1, 2, 2], [3, 3, 4, 4]]`。
3. 用 `np.clip` 把 `np.arange(20)` 中大于 15 的元素截断为 15、小于 5 的截断为 5。

> 👉 **下一章：`04_形状变换与数组拼接.ipynb`**
> 学会了逐元素运算和广播，下一步就是「折腾形状」：reshape、转置、拼接、拆分……
> 掌握它们，你才能自由地把数据摆成任何想要的形状。